# Phase II — Lindsey Features on Karin's Pantip Data (All 3 Experiments)

**Goal.** Replace Karin's Phase II feature extraction with Lindsey's 12-dimension behavioral scoring. Score each comment, aggregate to user level (62 features), and emit per-experiment train/test CSVs that drop straight into Phase III.

**Inputs** (under `./Karin_dataset/`):
- `00_Datasets.txt` — full post records (JSON, UTF-8-BOM)
- `03_First_Train_Sock.txt`, `03_First_Train_Normal.txt`, `03_First_Test_Sock.txt`, `03_First_Test_Normal.txt` — Exp 1 (~30/70)
- `04_Second_Train_Sock.txt`, `04_Second_Train_Normal.txt`, `04_Second_Test_Sock.txt`, `04_Second_Test_Normal.txt` — Exp 2 (50/50)
- `05_Third_Train_Sock.txt`, `05_Third_Train_Normal.txt`, `05_Third_Test_Sock.txt`, `05_Third_Test_Normal.txt` — Exp 3 (~70/30)

Each split file is JSON of the form `List[List[user_id]]` — groups of users sharing IPs (sockpuppet groups) or paired controls (normal groups). We flatten groups → users → label = 1 (sock) or 0 (normal).

**Outputs** (per experiment `e ∈ {1,2,3}`):
- `phase2_train_exp{e}.csv` — user-level 62-feature train matrix
- `phase2_test_exp{e}.csv` — user-level 62-feature test matrix

Content-sensitive dimensions (2 political, 3 attack, 4 emotional, 5 low-effort, 6 coordination, 7 algorithmic-gaming, 8 manufactured-authenticity) use Thai/Pantip-adapted term lists. Structural dimensions (1 emoji, 9 length, 10 repetition, 11 punctuation, 12 caps) are language-agnostic.

## Cell 1 — Imports and config

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # we don't need GPU here; suppresses cuDNN noise

import json
import re
import unicodedata
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd

DATA_DIR = Path('./Karin_dataset')
OUT_DIR  = Path('.')
OUT_DIR.mkdir(exist_ok=True)

POSTS_FILE = DATA_DIR / '00_Datasets.txt'

SPLIT_FILES = {
    1: {
        'train_sock':   DATA_DIR / '03_First_Train_Sock.txt',
        'train_normal': DATA_DIR / '03_First_Train_Normal.txt',
        'test_sock':    DATA_DIR / '03_First_Test_Sock.txt',
        'test_normal':  DATA_DIR / '03_First_Test_Normal.txt',
    },
    2: {
        'train_sock':   DATA_DIR / '04_Second_Train_Sock.txt',
        'train_normal': DATA_DIR / '04_Second_Train_Normal.txt',
        'test_sock':    DATA_DIR / '04_Second_Test_Sock.txt',
        'test_normal':  DATA_DIR / '04_Second_Test_Normal.txt',
    },
    3: {
        'train_sock':   DATA_DIR / '05_Third_Train_Sock.txt',
        'train_normal': DATA_DIR / '05_Third_Train_Normal.txt',
        'test_sock':    DATA_DIR / '05_Third_Test_Sock.txt',
        'test_normal':  DATA_DIR / '05_Third_Test_Normal.txt',
    },
}

print('Config OK')
print('Data dir:', DATA_DIR.resolve())


Config OK
Data dir: /home/Ken/research/MyWork/20260526-4/Karin_dataset


## Cell 2 — Load posts and labeled users per experiment

In [2]:
def load_json_bom(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        return json.load(f)

# --- Posts -------------------------------------------------------------------
records = load_json_bom(POSTS_FILE)
posts = pd.json_normalize(records)

# Datetime parse — Karin's format is m/d/Y H:M:S
posts['datetime'] = pd.to_datetime(posts['datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
posts['user_id']  = posts['user_id'].astype(int)
posts['thread_id'] = posts['thread_id'].astype(int)

# Combined text for scoring: thread_name + message gives richer signal
if 'thread_name' in posts.columns:
    posts['text'] = (posts['thread_name'].fillna('').astype(str) + ' ' +
                     posts['messages'].fillna('').astype(str)).str.strip()
else:
    posts['text'] = posts['messages'].fillna('').astype(str).str.strip()

# Engagement signal (Pantip "helpful" upvotes); fill missing
for col in ('helpful', 'likeScore', 'emo_like'):
    if col not in posts.columns:
        posts[col] = 0
posts['helpful'] = pd.to_numeric(posts['helpful'], errors='coerce').fillna(0)

print(f'Posts:   {len(posts):,}')
print(f'Users:   {posts.user_id.nunique():,}')
print(f'Threads: {posts.thread_id.nunique():,}')

# --- Split files -------------------------------------------------------------
def load_user_label_map(exp_id):
    files = SPLIT_FILES[exp_id]
    train_sock_groups   = load_json_bom(files['train_sock'])
    train_normal_groups = load_json_bom(files['train_normal'])
    test_sock_groups    = load_json_bom(files['test_sock'])
    test_normal_groups  = load_json_bom(files['test_normal'])

    flatten = lambda groups: [int(u) for g in groups for u in g]

    train_users = {u: 1 for u in flatten(train_sock_groups)}
    train_users.update({u: 0 for u in flatten(train_normal_groups)})
    test_users  = {u: 1 for u in flatten(test_sock_groups)}
    test_users.update({u: 0 for u in flatten(test_normal_groups)})

    # Sanity: no leakage between train and test
    leak = set(train_users) & set(test_users)
    assert not leak, f'Exp {exp_id}: user leak between train and test: {leak}'
    return train_users, test_users

for e in (1, 2, 3):
    tr, te = load_user_label_map(e)
    print(f'Exp {e}: train={len(tr)} (sock={sum(tr.values())}) | test={len(te)} (sock={sum(te.values())})')


Posts:   95,904
Users:   12,954
Threads: 6,064
Exp 1: train=86 (sock=43) | test=194 (sock=97)
Exp 2: train=140 (sock=70) | test=140 (sock=70)
Exp 3: train=198 (sock=99) | test=82 (sock=41)


## Cell 3 — Lindsey's 12-dimension comment scoring (Thai-adapted)

Each comment yields 12 scores in `[0, 1]`. Term lists come from Karin's thesis Thai political context (2023 election: ก้าวไกล / เพื่อไทย / etc.).

In [3]:
import emoji as emoji_lib  # pip install emoji

# ── Thai-adapted term lists ──────────────────────────────────────────────────
THAI_POLITICAL = [
    'ก้าวไกล','พิธา','เพื่อไทย','เศรษฐา','ประยุทธ์','ประวิตร','อนุทิน',
    'รทสช','พปชร','ภูมิใจไทย','รวมไทยสร้างชาติ','พลังประชารัฐ',
    'เลือกตั้ง','การเมือง','นายกฯ','นายกรัฐมนตรี','รัฐบาล','ฝ่ายค้าน','ส.ส.','รัฐสภา',
]

THAI_ATTACK = {
    'ล้มล้าง': 0.9, 'ขายชาติ': 0.9,
    'สลิ่ม': 0.8, 'ไอ้โม่ง': 0.8, 'คอร์รัปชัน': 0.8, 'เผด็จการ': 0.8,
    'โกง': 0.7, 'นอมินี': 0.7,
    'เสื้อแดง': 0.6, 'ระบอบ': 0.6,
    'fake': 0.5,
}

THAI_EMOTIONAL = {
    'น้ำตาไหล': 0.9,
    'ซึ้งมาก': 0.8, 'ขนลุก': 0.8,
    'น่าเศร้า': 0.7, 'โกรธมาก': 0.7,
    'ทนไม่ได้': 0.6, 'น่าละอาย': 0.6, 'รักประเทศ': 0.6,
    'ภูมิใจ': 0.5, 'สุดยอด': 0.5,
}

LOW_EFFORT_PHRASES = ['เห็นด้วย','ถูกต้อง','ใช่เลย','เลยนะ','โคตรดี','สนับสนุน','เชียร์','แน่นอน','ปลื้ม']
SINGLE_NAME_PATTERN = re.compile(
    r'^(ก้าวไกล|เพื่อไทย|พิธา|เศรษฐา|ประยุทธ์|ประวิตร|อนุทิน|รทสช|พปชร)\s*[!\s]*\s*$'
)

DIRECT_TAGS = ['@ก้าวไกล','@เพื่อไทย','@พิธา','@เศรษฐา']
COORD_PHRASES = ['ทีมก้าวไกล','ทีมเพื่อไทย','มาเลย','ร่วมกัน','ขอเชิญชวน','ช่วยกันได้เลย','พวกเรา']

ALGO_GAMING = {
    'ช่วยกันแชร์': 1.0, 'แชร์ต่อด้วย': 1.0, 'ช่วยกันกระจาย': 1.0,
    'ไวรัล': 1.0, 'ติดเทรนด์': 1.0, 'viral': 1.0, 'trending': 1.0,
    'ช่วยกันกด': 0.7, 'ส่งต่อ': 0.7, 'ให้คนอื่นรู้': 0.7, 'share': 0.7, 'repost': 0.7,
}

AUTH_PHRASES = [
    'ไม่เคยคอมเมนต์','ไม่ค่อยสนใจการเมือง','ไม่ใช่คนการเมือง','เป็นคนกลางนะ',
    'ไม่ได้เลือกข้าง','แต่ต้องบอกว่า','ในฐานะคนไทย','ครั้งแรกที่โพสต์','as a',
]

PUNCT_CHARS = set('!?.,;:')

def count_emojis(text):
    return sum(1 for ch in text if ch in emoji_lib.EMOJI_DATA)

def word_count_thai(text):
    """Lightweight word count: split on whitespace + remove emoji.
    For Thai tokenization we approximate; structural dims don't need pythainlp precision."""
    no_emoji = ''.join(ch for ch in text if ch not in emoji_lib.EMOJI_DATA)
    parts = re.findall(r'\S+', no_emoji)
    return len(parts)

def score_emoji(text):
    n = count_emojis(text)
    wc = max(word_count_thai(text), 1)
    if n == 0:   base = 0.0
    elif n <= 2: base = 0.2
    elif n <= 4: base = 0.4
    elif n <= 6: base = 0.6
    elif n <= 9: base = 0.8
    else:        base = 1.0
    ratio = n / wc
    if ratio > 2:   base = min(1.0, base + 0.3)
    elif ratio > 1: base = min(1.0, base + 0.2)
    return base

def score_political(text):
    wc = max(word_count_thai(text), 1)
    hits = sum(text.count(t) for t in THAI_POLITICAL)
    return min(1.0, (hits / wc) * 5)

def score_max_dict(text, term_weights):
    best = 0.0
    for term, w in term_weights.items():
        if term in text and w > best:
            best = w
    return best

def score_attack(text):    return score_max_dict(text, THAI_ATTACK)
def score_emotional(text): return score_max_dict(text, THAI_EMOTIONAL)

def score_low_effort(text):
    wc = word_count_thai(text)
    if wc > 3:
        return 0.0
    if SINGLE_NAME_PATTERN.match(text.strip()):
        return 0.9
    if any(p in text for p in LOW_EFFORT_PHRASES):
        return 0.8
    return 0.0

def score_coordination(text):
    if any(tag in text for tag in DIRECT_TAGS):
        return 1.0
    if any(p in text for p in COORD_PHRASES):
        return 0.4
    return 0.0

def score_algorithmic(text): return score_max_dict(text, ALGO_GAMING)

def score_authenticity(text):
    return 0.8 if any(p in text for p in AUTH_PHRASES) else 0.0

def score_length(text):
    wc = word_count_thai(text)
    if wc == 0:  return 0.6
    if wc <= 2:  return 0.4
    if wc >= 50: return 0.3
    return 0.0

def score_repetition(text):
    s_punct = 0.4 if re.search(r'[!?]{3,}', text) else 0.0
    s_char  = 0.3 if re.search(r'(.)\1{3,}', text) else 0.0
    return max(s_punct, s_char)

def score_punctuation(text):
    wc = word_count_thai(text)
    if wc == 0:
        return 0.0
    n_punct = sum(1 for ch in text if ch in PUNCT_CHARS)
    ratio = n_punct / wc
    if ratio > 0.5: return 0.6
    if ratio > 0.3: return 0.3
    return 0.0

def score_caps(text):
    if len(text) == 0:
        return 0.0
    caps = sum(1 for ch in text if ch.isupper())
    ratio = caps / len(text)
    if ratio > 0.7 and len(text) > 10: return 0.8
    if ratio > 0.5:                    return 0.4
    return 0.0

DIM_NAMES = [
    'emoji','political','attack','emotional','low_effort','coordination',
    'algorithmic','authenticity','length','repetition','punctuation','caps',
]

DIM_FUNCS = [
    score_emoji, score_political, score_attack, score_emotional,
    score_low_effort, score_coordination, score_algorithmic, score_authenticity,
    score_length, score_repetition, score_punctuation, score_caps,
]

def analyze_comment(text):
    text = text if isinstance(text, str) else ''
    return {name: fn(text) for name, fn in zip(DIM_NAMES, DIM_FUNCS)}

# Smoke test
sample = 'ก้าวไกล! พิธาสู้ๆ ✊✊✊ #เลือกตั้ง66 ช่วยกันแชร์นะ'
print(analyze_comment(sample))


{'emoji': 0.4, 'political': 1.0, 'attack': 0.0, 'emotional': 0.0, 'low_effort': 0.0, 'coordination': 0.0, 'algorithmic': 1.0, 'authenticity': 0.0, 'length': 0.0, 'repetition': 0.0, 'punctuation': 0.0, 'caps': 0.0}


## Cell 4 — Score every comment

In [4]:
print('Scoring comments...')
scores_df = posts['text'].apply(analyze_comment).apply(pd.Series)
scored = pd.concat([posts[['user_id','thread_id','datetime','helpful','text']].reset_index(drop=True),
                    scores_df.reset_index(drop=True)], axis=1)
print(f'Scored {len(scored):,} comments across {scored.user_id.nunique():,} users')
scored.head(3)


Scoring comments...
Scored 95,904 comments across 12,954 users


,user_id,thread_id,datetime,helpful,text,emoji,political,attack,emotional,low_effort,coordination,algorithmic,authenticity,length,repetition,punctuation,caps
0,7464651,42111000,2023-07-10 09:19:04,0,ก้าวไกลคือแบบอย่างการเมืองสุจริต ก้าวไกลคือแบบ...,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7361167,42111000,2023-07-10 09:34:22,0,ก้าวไกลคือแบบอย่างการเมืองสุจริต พปชร รทสช ต่า...,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7576977,42111000,2023-07-10 09:35:04,0,ก้าวไกลคือแบบอย่างการเมืองสุจริต ขึ้น ค่าแรง 4...,0.0,0.588235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.0


## Cell 5 — Aggregate to user level (62 features)

For each of the 12 dimensions: `_mean`, `_max`, `_std`, `_sum` → 48 features.  
Plus 14 composite/engagement features → **62 total**.

In [5]:
def aggregate_user_features(df_user_posts):
    """Given the subset of `scored` belonging to one user, return a 62-feature dict."""
    feats = {}
    for dim in DIM_NAMES:
        s = df_user_posts[dim]
        feats[f'{dim}_mean'] = s.mean()
        feats[f'{dim}_max']  = s.max()
        feats[f'{dim}_std']  = s.std(ddof=0) if len(s) > 1 else 0.0
        feats[f'{dim}_sum']  = s.sum()

    # Composite scores across all dims, per comment
    per_comment_max = df_user_posts[DIM_NAMES].max(axis=1)
    feats['overall_max']   = per_comment_max.max() if len(per_comment_max) else 0.0
    feats['overall_mean']  = per_comment_max.mean() if len(per_comment_max) else 0.0
    feats['n_high_score']  = int((per_comment_max > 0.6).sum())
    feats['n_med_score']   = int(((per_comment_max >= 0.3) & (per_comment_max <= 0.6)).sum())

    # Engagement
    n_comments = len(df_user_posts)
    n_threads  = df_user_posts['thread_id'].nunique()
    feats['n_comments']      = n_comments
    feats['n_threads']       = n_threads
    feats['likes_total']     = df_user_posts['helpful'].sum()
    feats['likes_per_comment'] = (df_user_posts['helpful'].sum() / n_comments) if n_comments else 0.0
    feats['threads_per_comment'] = (n_threads / n_comments) if n_comments else 0.0

    # Temporal
    times = df_user_posts['datetime'].dropna().sort_values()
    if len(times) >= 2:
        deltas = times.diff().dt.total_seconds().dropna()
        feats['delta_mean_s'] = deltas.mean()
        feats['delta_std_s']  = deltas.std(ddof=0)
        feats['delta_min_s']  = deltas.min()
    else:
        feats['delta_mean_s'] = 0.0
        feats['delta_std_s']  = 0.0
        feats['delta_min_s']  = 0.0

    # Likes summary (mean/replies placeholders since Pantip has no reply-count)
    feats['likes_mean']   = df_user_posts['helpful'].mean() if n_comments else 0.0
    feats['replies_mean'] = 0.0   # placeholder — Pantip doesn't expose reply counts
    return feats

print('Aggregating per-user features...')
user_features = scored.groupby('user_id').apply(aggregate_user_features).apply(pd.Series)
user_features = user_features.reset_index()
print(f'Built feature matrix: {user_features.shape}  (expect 62 + user_id = 63 cols)')
print(user_features.columns.tolist())


Aggregating per-user features...


/tmp/ipykernel_13807/4197758449.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_features = scored.groupby('user_id').apply(aggregate_user_features).apply(pd.Series)


Built feature matrix: (12954, 63)  (expect 62 + user_id = 63 cols)
['user_id', 'emoji_mean', 'emoji_max', 'emoji_std', 'emoji_sum', 'political_mean', 'political_max', 'political_std', 'political_sum', 'attack_mean', 'attack_max', 'attack_std', 'attack_sum', 'emotional_mean', 'emotional_max', 'emotional_std', 'emotional_sum', 'low_effort_mean', 'low_effort_max', 'low_effort_std', 'low_effort_sum', 'coordination_mean', 'coordination_max', 'coordination_std', 'coordination_sum', 'algorithmic_mean', 'algorithmic_max', 'algorithmic_std', 'algorithmic_sum', 'authenticity_mean', 'authenticity_max', 'authenticity_std', 'authenticity_sum', 'length_mean', 'length_max', 'length_std', 'length_sum', 'repetition_mean', 'repetition_max', 'repetition_std', 'repetition_sum', 'punctuation_mean', 'punctuation_max', 'punctuation_std', 'punctuation_sum', 'caps_mean', 'caps_max', 'caps_std', 'caps_sum', 'overall_max', 'overall_mean', 'n_high_score', 'n_med_score', 'n_comments', 'n_threads', 'likes_total', '

## Cell 6 — Build train/test CSVs for each experiment

In [6]:
for exp_id in (1, 2, 3):
    train_map, test_map = load_user_label_map(exp_id)

    train_df = user_features[user_features['user_id'].isin(train_map.keys())].copy()
    train_df['label'] = train_df['user_id'].map(train_map).astype(int)

    test_df  = user_features[user_features['user_id'].isin(test_map.keys())].copy()
    test_df['label']  = test_df['user_id'].map(test_map).astype(int)

    # Fill any feature NaN with 0 (e.g. users with single comment → std undefined)
    feat_cols = [c for c in train_df.columns if c not in ('user_id', 'label')]
    train_df[feat_cols] = train_df[feat_cols].fillna(0)
    test_df[feat_cols]  = test_df[feat_cols].fillna(0)

    train_path = OUT_DIR / f'phase2_train_exp{exp_id}.csv'
    test_path  = OUT_DIR / f'phase2_test_exp{exp_id}.csv'
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print(f'Exp {exp_id}:  train {train_df.shape} ({train_df.label.sum()} sock)  '
          f'|  test {test_df.shape} ({test_df.label.sum()} sock)')
    print(f'         → {train_path}')
    print(f'         → {test_path}')


Exp 1:  train (86, 64) (43 sock)  |  test (194, 64) (97 sock)
         → phase2_train_exp1.csv
         → phase2_test_exp1.csv
Exp 2:  train (140, 64) (70 sock)  |  test (140, 64) (70 sock)
         → phase2_train_exp2.csv
         → phase2_test_exp2.csv
Exp 3:  train (198, 64) (99 sock)  |  test (82, 64) (41 sock)
         → phase2_train_exp3.csv
         → phase2_test_exp3.csv


## Done
Move on to `phase3_all_experiments.ipynb`, which loads these six CSVs and runs the five classifiers.